In [24]:
import pandas as pd

def naive_bayers_predict(data, query):
  # create DataFrame
  df = pd.DataFrame(data)

  # calculate class priors: P(Yes) and P(No)
  total_count = len(df)
  class_counts = df['PlayTennis'].value_counts()
  p_yes = class_counts['yes'] / total_count
  p_no = class_counts['no'] / total_count

  print(f"---Priors---")
  print(f"p(yes):{p_yes:.4f}")
  print(f"p(no):{p_no:.4f}")

  # 2. Calculate Likelihoods and Posterior scores
  # Score = Prior * Product(Likelihoods)

  posterior_yes = p_yes
  posterior_no = p_no

  print(f"\n --- Likelihoods ---")
  for Features, value in query.items():
    # Calculate P(Feature=Value | Yes)
    yes_subset = df[df['PlayTennis'] == 'yes']
    feat_given_yes = yes_subset[yes_subset[Features] == value].shape[0] / class_counts['yes']

    # Calculate P(Feature=Value | No)
    no_subset = df[df['PlayTennis'] == 'no']
    feat_given_no = no_subset[no_subset[Features] == value].shape[0] / class_counts['no']

    # Update posteriors
    posterior_yes *= feat_given_yes
    posterior_no *= feat_given_no

    print(f"P({Features}={value}|Yes):{feat_given_yes:.4f}, P({Features}={value}|No):{feat_given_no:.4f}")

  # 3. Final Prediction
  print(f"\n---Posterior Scores (unnormalized)---")
  print(f"Score(Yes): {posterior_yes:.4f}")
  print(f"Score(No): {posterior_no:.4f}")

  # Normalize to get percentages
  evidence = posterior_yes + posterior_no
  # Check for zero evidence to prevent division by zero
  if evidence == 0:
      prob_yes = 0.0
      prob_no = 0.0
  else:
      prob_yes = posterior_yes / evidence
      prob_no = posterior_no / evidence

  print(f"\n---Final Probabilities---")
  print(f"Probability(Yes): {prob_yes:.2%}")
  print(f"Probability(No): {prob_no:.2%}")

  return "Yes" if prob_yes > prob_no else "No"

# --- Main Execution ---

# Data Definition (corrected to have consistent lengths and 'PlayTennis' column)
dataset = {
    'Day': ['d1','d2','d3','d4','d5','d6','d7','d8','d9','d10','d11','d12','d13','d14'],
    'Outlook': ['sunny','sunny','overcast','rain','rain','rain','overcast','sunny','sunny','rain','sunny','overcast','rain', 'overcast'],
    'Temperature': ['hot','hot','hot','mild','cool','cool','cool','mild','cool','mild', 'mild', 'hot', 'hot', 'hot'],
    'Humidity': ['high','high','high','normal','normal','normal', 'high', 'normal', 'high', 'normal', 'high', 'normal', 'high', 'normal'],
    'Wind': ['weak','strong','weak','weak','weak','strong','strong','weak','weak','weak','strong','strong','weak','strong'],
    'PlayTennis': ['no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'no']

}

#New Instance to predict
new_instance = {
    'Outlook': 'sunny',
    'Temperature': 'cool',
    'Humidity': 'high',
    'Wind': 'strong'
}

# Predict
prediction = naive_bayers_predict(dataset, new_instance)
print(f"\nPredicted Class: {prediction}")

---Priors---
p(yes):0.6429
p(no):0.3571

 --- Likelihoods ---
P(Outlook=sunny|Yes):0.2222, P(Outlook=sunny|No):0.6000
P(Temperature=cool|Yes):0.3333, P(Temperature=cool|No):0.2000
P(Humidity=high|Yes):0.5556, P(Humidity=high|No):0.4000
P(Wind=strong|Yes):0.3333, P(Wind=strong|No):0.6000

---Posterior Scores (unnormalized)---
Score(Yes): 0.0088
Score(No): 0.0103

---Final Probabilities---
Probability(Yes): 46.16%
Probability(No): 53.84%

Predicted Class: No
